# Fine-tune BART on MM-AVS (text only)

Fine-tunes `facebook/bart-large-cnn` on the MM-AVS train split, picks the best epoch on the
validation split, then evaluates **pretrained vs fine-tuned BART on the same held-out test stories**.

**Before running:**
1. *Add Input* → attach the private dataset **mmavs-text**.
2. *Settings* → **Accelerator: GPU T4** (single T4 is enough) and **Internet: On** (needed to download BART).
3. Run once with `PILOT = True` (~5 min) to check everything works, then set `PILOT = False` and *Save Version → Save & Run All*.

Outputs land in `/kaggle/working/`: best checkpoint, per-example predictions, and `results.json`.

In [ ]:
!pip install -q rouge-score

In [ ]:
import glob, json, math, os, random, re, time
from collections import Counter
from pathlib import Path

import numpy as np
import torch
import transformers
from torch.utils.data import DataLoader
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, get_linear_schedule_with_warmup
from rouge_score import rouge_scorer

# ---- configuration -------------------------------------------------------
PILOT = False                 # True: tiny smoke test. False: full run.

MODEL_NAME = "facebook/bart-large-cnn"
SEED = 42
MAX_SOURCE_LEN = 1024         # BART's limit; ~25% of articles are longer and get truncated
MAX_TARGET_LEN = 192          # covers ~99% of reference summaries
NUM_EPOCHS = 6                # best epoch is chosen on validation, not assumed
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
BATCH_SIZE = 2
GRAD_ACCUM = 4                # effective batch size 8
GRADIENT_CHECKPOINTING = True # trades ~30% speed for memory headroom on a 16 GB T4
TRAIN_ON_REPS_ONLY = True     # one copy per duplicate story group

# Fixed before seeing any test result; identical for pretrained and fine-tuned models.
GEN_KWARGS = dict(num_beams=4, no_repeat_ngram_size=3, length_penalty=2.0,
                  min_new_tokens=20, max_new_tokens=160, early_stopping=True)
EVAL_BATCH_SIZE = 8

OUT_DIR = Path("/kaggle/working")
BEST_DIR = OUT_DIR / "bart-mmavs-best"

if PILOT:
    NUM_EPOCHS = 1

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")
print("torch", torch.__version__, "| transformers", transformers.__version__)

## Load the splits

In [ ]:
data_dir = Path(glob.glob("/kaggle/input/**/train.jsonl", recursive=True)[0]).parent
print("data dir:", data_dir)

raw = {split: [json.loads(line) for line in open(data_dir / f"{split}.jsonl")]
       for split in ("train", "val", "test")}

# MM-AVS Daily Mail highlights are glued together with no separator at all
# ("...last FridayRaiders seen on CCTV..."). Split where a lowercase letter, digit
# or closing quote runs into a capitalised word, unless the boundary falls inside a
# camelCase word that also appears in the (correctly spaced) articles, e.g.
# iPhone, YouTube, DailyMailTV, or a Mc/Mac surname.
REAL_CAMEL = {w for w, c in Counter(
    w for rows in raw.values() for r in rows
    for w in re.findall(r"[A-Za-z]*[a-z][A-Z][A-Za-z]*", r["article"])).items() if c >= 2}
# A quote only counts as a highlight end when it closes a word ("...said'Police").
GLUED = re.compile(r"(?:(?<=[a-z0-9%)])|(?<=[\w,]['\u2019\"]))(?=[A-Z][a-z]|[AI]\b|[A-Z]{2,}\b)")
GLUED_STOP = re.compile(r"(?:(?<=[a-z0-9%)'\u2019\" ][.!?]))(?=[A-Z\"'\u2018])")   # "...one.Parental", "...one .Parental"

def inside_real_word(text, pos):
    a, b = pos, pos
    while a > 0 and text[a - 1].isalpha():
        a -= 1
    while b < len(text) and text[b].isalpha():
        b += 1
    run, k = text[a:b], pos - a
    if run[max(0, k - 3):k] == "Mac" or run[max(0, k - 2):k] == "Mc":
        return True
    return any(run[i:j] in REAL_CAMEL for i in range(k) for j in range(k + 1, len(run) + 1))

def split_highlights(text):
    text = GLUED_STOP.sub("\n", text.replace("\xa0", " "))
    cuts = [m.start() for m in GLUED.finditer(text) if not inside_real_word(text, m.start())]
    for pos in reversed(cuts):
        text = text[:pos] + "\n" + text[pos:]
    parts = []
    for s in (s.strip() for s in text.split("\n")):
        if parts and re.fullmatch(r"[\'\u2019\"]+", s):
            parts[-1] += s          # closing quote split off its sentence ("...said.'")
        elif s:
            parts.append(s)
    return parts

def clean_summary(text):
    # One highlight per sentence, each ending in a full stop, so the model learns
    # well-formed sentences and rougeLsum can split them.
    highlights = [re.sub(r"\s+", " ", h) for h in split_highlights(text)]
    return " ".join(h if h[-1] in ".!?" else h + "." for h in highlights)

# Articles have the same scraping damage: sentences run together ("...the sideline.Feki was..."),
# non-breaking spaces, a glued "(CNN)" dateline and a few CNN boilerplate lines.
GLUED_SENTENCE = re.compile(r"(?<=[a-z0-9%)][.!?])(?=[A-Z\u2018\u201c])|(?<=[a-z0-9%)][.!?][\u2019\u201d'\"])(?=[A-Z])")
BOILERPLATE = re.compile(r"Copyright \d{4} The Conversation\.|Some rights reserved\.|"
                         r"Share your thoughts with [^.]{1,60} on Twitter @\w+\.|"
                         r"(?<=[.!?] )[A-Z][^.!?]{1,80} contributed to this report\.")

def clean_article(text):
    text = re.sub(r"^\s*\(CNN\)\s*(--)?\s*", "", text.replace("\xa0", " "))
    text = BOILERPLATE.sub("", GLUED_SENTENCE.sub(" ", text))
    return re.sub(r"\s+", " ", text).strip()

def load(split):
    rows = [dict(r) for r in raw[split]]
    for r in rows:
        r["article"] = clean_article(r["article"])
        r["summary"] = clean_summary(r["summary"])
    return rows

train_rows = load("train")
val_rows = [r for r in load("val") if r["is_group_rep"]]
test_rows = [r for r in load("test") if r["is_group_rep"]]   # each test story counted once
if TRAIN_ON_REPS_ONLY:
    train_rows = [r for r in train_rows if r["is_group_rep"]]

if PILOT:
    train_rows, val_rows, test_rows = train_rows[:48], val_rows[:16], test_rows[:16]

print(f"train {len(train_rows)} | val {len(val_rows)} | test {len(test_rows)}")
print(json.load(open(data_dir / "splits_summary.json"))["counts"])
print("example cleaned reference:", test_rows[-1]["summary"][:300])

## Tokenize

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def encode(rows):
    enc = tokenizer([r["article"] for r in rows], max_length=MAX_SOURCE_LEN, truncation=True)
    labels = tokenizer(text_target=[r["summary"] for r in rows], max_length=MAX_TARGET_LEN, truncation=True)
    return [{"input_ids": enc["input_ids"][i], "labels": labels["input_ids"][i]} for i in range(len(rows))]

def truncation_report(rows, name):
    a = [len(x) for x in tokenizer([r["article"] for r in rows])["input_ids"]]
    s = [len(x) for x in tokenizer(text_target=[r["summary"] for r in rows])["input_ids"]]
    print(f"{name}: articles > {MAX_SOURCE_LEN} tokens: {np.mean(np.array(a) > MAX_SOURCE_LEN):.1%} | "
          f"summaries > {MAX_TARGET_LEN} tokens: {np.mean(np.array(s) > MAX_TARGET_LEN):.1%}")

truncation_report(train_rows, "train")
train_data, val_data = encode(train_rows), encode(val_rows)

def collate(batch):
    pad = tokenizer.pad_token_id
    src_len = max(len(b["input_ids"]) for b in batch)
    tgt_len = max(len(b["labels"]) for b in batch)
    input_ids = torch.full((len(batch), src_len), pad)
    attention = torch.zeros((len(batch), src_len), dtype=torch.long)
    labels = torch.full((len(batch), tgt_len), -100)   # -100 = ignored by the loss
    for i, b in enumerate(batch):
        input_ids[i, :len(b["input_ids"])] = torch.tensor(b["input_ids"])
        attention[i, :len(b["input_ids"])] = 1
        labels[i, :len(b["labels"])] = torch.tensor(b["labels"])
    return {"input_ids": input_ids, "attention_mask": attention, "labels": labels}

g = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate, generator=g)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, collate_fn=collate)

## Generation and ROUGE helpers

In [ ]:
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL", "rougeLsum"], use_stemmer=True)
METRICS = ["rouge1", "rouge2", "rougeL", "rougeLsum"]

def sentences_per_line(text):
    # rougeLsum compares sentence by sentence; it expects one sentence per line.
    return "\n".join(s for s in re.split(r"(?<=[.!?])\s+", text.strip()) if s)

@torch.no_grad()
def generate(model, rows):
    model.eval()
    preds = []
    for i in range(0, len(rows), EVAL_BATCH_SIZE):
        batch = rows[i:i + EVAL_BATCH_SIZE]
        enc = tokenizer([r["article"] for r in batch], max_length=MAX_SOURCE_LEN,
                        truncation=True, padding=True, return_tensors="pt").to(device)
        with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
            out = model.generate(**enc, **GEN_KWARGS)
        preds += tokenizer.batch_decode(out, skip_special_tokens=True)
    return [p.strip() for p in preds]

def score(rows, preds):
    per_example = []
    for r, p in zip(rows, preds):
        s = scorer.score(sentences_per_line(r["summary"]), sentences_per_line(p))
        per_example.append({m: s[m].fmeasure for m in METRICS})
    return per_example

def summarize(per_example, n_boot=1000):
    # Mean F1 with a 95% bootstrap confidence interval over test stories.
    rng = np.random.default_rng(SEED)
    out = {}
    for m in METRICS:
        v = np.array([e[m] for e in per_example])
        boots = [rng.choice(v, len(v)).mean() for _ in range(n_boot)]
        out[m] = {"mean": float(v.mean()), "ci95": [float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))]}
    return out

def rouge_mean(summary):
    return float(np.mean([summary[m]["mean"] for m in ("rouge1", "rouge2", "rougeL")]))

## 1. Pretrained baseline on the test set

Run **before** fine-tuning, with the exact generation settings used later.

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)

t0 = time.time()
pre_preds = generate(model, test_rows)
pre_scores = score(test_rows, pre_preds)
pre_summary = summarize(pre_scores)
pre_minutes = (time.time() - t0) / 60
print(f"pretrained test ({len(test_rows)} stories, {pre_minutes:.1f} min)")
for m in METRICS:
    print(f"  {m:<9} {pre_summary[m]['mean']:.4f}  95% CI {pre_summary[m]['ci95'][0]:.4f}-{pre_summary[m]['ci95'][1]:.4f}")

## 2. Fine-tune, selecting the best epoch on validation

In [ ]:
if GRADIENT_CHECKPOINTING:
    model.gradient_checkpointing_enable()
    model.config.use_cache = False

no_decay = ["bias", "layer_norm.weight", "layernorm_embedding.weight"]
params = [
    {"params": [p for n, p in model.named_parameters() if not any(k in n for k in no_decay)], "weight_decay": WEIGHT_DECAY},
    {"params": [p for n, p in model.named_parameters() if any(k in n for k in no_decay)], "weight_decay": 0.0},
]
optimizer = torch.optim.AdamW(params, lr=LEARNING_RATE)
total_steps = math.ceil(len(train_loader) / GRAD_ACCUM) * NUM_EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, int(WARMUP_RATIO * total_steps), total_steps)
scaler = torch.amp.GradScaler(enabled=device.type == "cuda")

@torch.no_grad()
def val_loss(model):
    model.eval()
    losses = []
    for batch in val_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
            losses.append(model(**batch).loss.item())
    return float(np.mean(losses))

history, best = [], {"rouge_mean": -1.0, "epoch": None}
train_start = time.time()
if device.type == "cuda":
    torch.cuda.reset_peak_memory_stats()

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    running, t0 = [], time.time()
    optimizer.zero_grad()
    for step, batch in enumerate(train_loader, 1):
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
            loss = model(**batch).loss / GRAD_ACCUM
        scaler.scale(loss).backward()
        running.append(loss.item() * GRAD_ACCUM)
        if step % GRAD_ACCUM == 0 or step == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
        if step % 100 == 0:
            print(f"  epoch {epoch} step {step}/{len(train_loader)} loss {np.mean(running[-100:]):.4f}")

    model.config.use_cache = True
    v_loss = val_loss(model)
    v_preds = generate(model, val_rows)
    v_summary = summarize(score(val_rows, v_preds), n_boot=1)
    if GRADIENT_CHECKPOINTING:
        model.config.use_cache = False

    record = {"epoch": epoch, "train_loss": float(np.mean(running)), "val_loss": v_loss,
              "val_rouge": {m: v_summary[m]["mean"] for m in METRICS},
              "val_rouge_mean": rouge_mean(v_summary), "minutes": (time.time() - t0) / 60}
    history.append(record)
    print(f"epoch {epoch}: train_loss {record['train_loss']:.4f} | val_loss {v_loss:.4f} | "
          f"val R1/R2/RL {v_summary['rouge1']['mean']:.4f}/{v_summary['rouge2']['mean']:.4f}/"
          f"{v_summary['rougeL']['mean']:.4f} | {record['minutes']:.1f} min")

    if record["val_rouge_mean"] > best["rouge_mean"]:
        best = {"rouge_mean": record["val_rouge_mean"], "epoch": epoch}
        best_val_preds = v_preds
        model.save_pretrained(BEST_DIR)
        tokenizer.save_pretrained(BEST_DIR)
        print(f"  -> new best, saved to {BEST_DIR}")

train_minutes = (time.time() - train_start) / 60
peak_gb = torch.cuda.max_memory_allocated() / 1e9 if device.type == "cuda" else None
print(f"best epoch {best['epoch']} | training {train_minutes:.1f} min | peak GPU memory {peak_gb:.1f} GB")

## 3. Fine-tuned model on the same test set

In [ ]:
del model, optimizer
torch.cuda.empty_cache()
model = AutoModelForSeq2SeqLM.from_pretrained(BEST_DIR).to(device)   # reload what was actually saved

t0 = time.time()
ft_preds = generate(model, test_rows)
ft_scores = score(test_rows, ft_preds)
ft_summary = summarize(ft_scores)
print(f"fine-tuned test ({len(test_rows)} stories, {(time.time() - t0) / 60:.1f} min)")
print(f"{'metric':<10}{'pretrained':>12}{'fine-tuned':>12}{'diff':>9}")
for m in METRICS:
    a, b = pre_summary[m]["mean"], ft_summary[m]["mean"]
    print(f"{m:<10}{a:>12.4f}{b:>12.4f}{b - a:>+9.4f}")

# Paired bootstrap: how often does fine-tuning beat pretrained when resampling test stories?
rng = np.random.default_rng(SEED)
diffs = np.array([[f[m] - p[m] for m in METRICS] for p, f in zip(pre_scores, ft_scores)])
wins = np.mean([diffs[rng.integers(0, len(diffs), len(diffs))].mean(0) > 0 for _ in range(1000)], axis=0)
paired = {m: float(w) for m, w in zip(METRICS, wins)}
print("bootstrap P(fine-tuned > pretrained):", {m: round(w, 3) for m, w in paired.items()})

## 4. Save predictions and results

In [ ]:
import csv

with open(OUT_DIR / "test_predictions.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["id", "source", "reference", "pretrained", "finetuned"]
               + [f"pre_{m}" for m in METRICS] + [f"ft_{m}" for m in METRICS])
    for r, p, ft, ps, fs in zip(test_rows, pre_preds, ft_preds, pre_scores, ft_scores):
        w.writerow([r["id"], r["source"], r["summary"], p, ft]
                   + [round(ps[m], 4) for m in METRICS] + [round(fs[m], 4) for m in METRICS])

with open(OUT_DIR / "val_predictions.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["id", "source", "reference", "finetuned"])
    for r, ft in zip(val_rows, best_val_preds):
        w.writerow([r["id"], r["source"], r["summary"], ft])

# The cleaned text the models actually saw, so later (image/audio/video) stages reuse it exactly.
for split, rows in (("train", train_rows), ("val", val_rows), ("test", test_rows)):
    with open(OUT_DIR / f"clean_{split}.jsonl", "w") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

results = {
    "pilot": PILOT,
    "model": MODEL_NAME,
    "versions": {"torch": torch.__version__, "transformers": transformers.__version__},
    "gpu": torch.cuda.get_device_name(0) if device.type == "cuda" else "cpu",
    "data": {"train": len(train_rows), "val": len(val_rows), "test": len(test_rows),
             "train_on_reps_only": TRAIN_ON_REPS_ONLY, "eval_on_reps_only": True},
    "hyperparameters": {"seed": SEED, "max_source_len": MAX_SOURCE_LEN, "max_target_len": MAX_TARGET_LEN,
                        "epochs": NUM_EPOCHS, "lr": LEARNING_RATE, "weight_decay": WEIGHT_DECAY,
                        "warmup_ratio": WARMUP_RATIO, "batch_size": BATCH_SIZE, "grad_accum": GRAD_ACCUM,
                        "gradient_checkpointing": GRADIENT_CHECKPOINTING, "fp16": True},
    "generation": GEN_KWARGS,
    "selection": "best epoch by mean of val ROUGE-1/2/L F1",
    "best_epoch": best["epoch"],
    "history": history,
    "training_minutes": train_minutes,
    "peak_gpu_memory_gb": peak_gb,
    "test": {"pretrained": pre_summary, "finetuned": ft_summary,
             "paired_bootstrap_p_finetuned_better": paired},
}
(OUT_DIR / "results.json").write_text(json.dumps(results, indent=2))
print("saved:", sorted(p.name for p in OUT_DIR.iterdir()))

## Notes for the report
- Test = official MM-AVS Daily Mail test IDs (+ their duplicate copies, scored once per story) and a held-out CNN slice.
- Generation settings were fixed before any test evaluation and are identical for both models.
- ROUGE measures word overlap with the reference, not factual correctness. Read `test_predictions.csv` for omissions and unsupported claims.
- `bart-large-cnn` was pretrained on CNN/DailyMail, which may overlap with MM-AVS articles, so the pretrained baseline may be optimistic.
- MM-AVS Daily Mail highlights are stored glued together with no separator; `clean_summary` re-splits them heuristically (camelCase words seen in articles are kept). Highlights joined only by a plain space cannot be recovered and stay as one sentence. The same cleaned references are used for both models.
- Articles are cleaned the same way for both models (`clean_article`): glued sentence boundaries re-spaced, non-breaking spaces normalised, the "(CNN)" dateline and a few CNN boilerplate lines removed. `clean_{train,val,test}.jsonl` hold the exact text used.
- Generation settings were kept identical to earlier runs even though ~6% of fine-tuned outputs hit `max_new_tokens`; changing them after seeing test outputs would be tuning on test.
- Single seed (42); differences between models are assessed with a paired bootstrap over test stories, not across seeds.
